# 04 - Federated Training Smoke Test

Run a **small** federated simulation (3 clients, 2 rounds, 60 consumers) using the same client, privacy, communication, aggregation, and simulator classes as the full suite. This is a code-path demonstration, not the reported 8-round result.

In [ ]:
from pathlib import Path
import json
import sys

# Headless-safe plotting; Jupyter still renders figures inline.
import matplotlib
matplotlib.use("Agg")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
import copy
import pandas as pd
from utilities.config import load_config
from utilities.seed import set_seed
from preprocessing.pipeline import prepare_data
from federated.partition import build_partition
from federated.simulator import FederatedSimulator
from privacy.base import build_privacy
from models.lightweight import build_lightweight

cfg = load_config("quick", [
    "data.synthetic.consumers=60",
    "data.synthetic.days=90",
    "data.synthetic.theft_rate=0.16",
    "clients.num_clients=3",
    "federated.rounds=2",
    "federated.local_epochs=2",
    "privacy.noise_multiplier=0.001",
    "communication.method=quantize",
])
set_seed(42)
splits = prepare_data(cfg, seed=42, log=print)
partitions = build_partition(splits.train, cfg, seed=42)
n_features = splits.train.X_matrix.shape[1]


## Run two rounds

In [ ]:
model_factory = lambda: build_lightweight(cfg, n_features, seed=42)
sim = FederatedSimulator(
    cfg=cfg,
    train_X=splits.train.X_matrix,
    train_y=splits.train.y,
    partitions=partitions,
    model_factory=model_factory,
    privacy_mechanism=build_privacy(cfg, seed=42),
    val_X=splits.val.X_matrix,
    val_y=splits.val.y,
    seed=42,
)
result = sim.run(rounds=2)
history = pd.DataFrame(result["history"])
val_metrics = pd.json_normalize(history["global_metrics"])
round_table = pd.concat([
    history[["round", "train_loss", "mean_val_loss", "bytes_upload", "bytes_download"]],
    val_metrics[["f1", "roc_auc"]].rename(columns={"f1": "val_f1", "roc_auc": "val_roc_auc"}),
], axis=1)
display(round_table)  # noqa: F821
print("Best validation F1:", result["best_val_f1"], "at round", result["best_round"])
print("Measured total bytes:", result["communication"]["total_bytes"])
print("Weight diagnostics:", result["history"][-1].get("weight_diagnostics"))


## What this demonstrates

- clients train locally and return only `ClientUpdate` deltas/statistics
- clipping + Gaussian noise are applied before aggregation
- int8 payloads are serialized and measured
- adaptive weights are normalized

The reported final result uses 5 clients x 8 rounds x 240 consumers; do not compare these two-round smoke numbers to it.